In [ ]:
# code to calculate the projections for maintenance and propoagation
#
#
# Version 2 Oct 17, 2024
# A. Thornton, A. Aiyyer
# NCSU Tropical and Large Scale Dynamics
#
#

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
# Any import of metpy will activate the accessors
from metpy.units import units
import geocat.viz as gv
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


In [ ]:
term_labels = [
        r'$\bar{u}\frac{\partial h^{\prime}}{\partial x}$',
        r'$\bar{v}\frac{\partial h^{\prime}}{\partial y}$',    
        r'$\bar{\omega}\frac{\partial h^{\prime}}{\partial p}$',
        r"$u^{\prime} \frac{\partial \bar{h}}{\partial x}$",
        r"$v^{\prime} \frac{\partial \bar{h}}{\partial y}$",
        r"${\omega}^{\prime} \frac{\partial \bar{h}}{\partial p}$",
        r"$u^{\prime} \frac{\partial h^{\prime}}{\partial x}$",
        r"$v^{\prime} \frac{\partial h^{\prime}}{\partial y}$",
        r"$\omega^{\prime} \frac{\partial h^{\prime}}{\partial p}$",
        r'SSHF',
        r'SLHF',
        r'Q',
        r'Res'
    ]

In [ ]:
def var_list(EnerVar):
    variables = [
        'ubar_'+EnerVar+'p_dx_vint',
        'vbar_'+EnerVar+'p_dy_vint',
        'wbar_'+EnerVar+'p_dp_vint',
        'up_'+EnerVar+'bar_dx_vint',
        'vp_' + EnerVar+'bar_dy_vint', 'wp_' + EnerVar+'bar_dp_vint',
        'up_'+EnerVar+'p_dx_vint',
        'vp_' + EnerVar+'p_dy_vint', 'wp_' + EnerVar+'p_dp_vint',
        'sshf', 'slhf',
        'Q_prime']
    return variables
def name_list(EnerVar):
    varNames =  [
        'ubar_'+ EnerVar+'p_dx_vint_prime', 
        'vbar_'+ EnerVar+'p_dy_vint_prime', 
        'wbar_'+ EnerVar+'p_dp_vint_prime',
        'up_' + EnerVar+'bar_dx_vint_prime', 
        'vp_' + EnerVar+'bar_dy_vint_prime', 'wp_' + EnerVar+'bar_dp_vint_prime',
        'up_' + EnerVar+'p_dx_vint_prime', 
        'vp_' + EnerVar+'p_dy_vint_prime', 'wp_' + EnerVar+'p_dp_vint_prime',
        'SSHF_prime', 'SLHF_prime', 
        'Q_prime']
    return varNames

In [ ]:
def plotting():
    # Negative bars in one color, positive in another
    def set_bar_color(value):
        if value >= 0:
            return  '///'  #None  
        else:
            return '///'  

    # Plot the total bar
    for i in range(len(term_labels)):
        result = h_proj[i]
        hatch = set_bar_color(result)
        ax[iplot,j].bar(i, result, color='white', edgecolor='black', alpha=0.7, hatch=hatch, capstyle='projecting', label='MSE')
    
    # Create a figure and axis object for plotting
    bar_width = 0.34  
    category_positions = range(len(term_labels))
    
    #Plot Bar 1 on the left half of the total bar
    ax[iplot,j].bar([pos - bar_width / 2 for pos in category_positions], s_proj, color='gold', width=bar_width, label='DSE', alpha=0.6)#, edgecolor='black')
    
    # Plot Bar 2 on the right half of the total bar
    ax[iplot,j].bar([pos + bar_width / 2 for pos in category_positions], q_proj, color='lightskyblue', width=bar_width, label='Moisture', alpha=0.7)#, edgecolor='black')
    
    
    # Set x-ticks and labels for ocean terms
    ax[iplot,j].set_xticks(np.arange(len(term_labels)))
    ax[iplot,j].set_xticklabels(term_labels, rotation='horizontal', fontsize=14)
    Rtitle = 'Lat=' + str(latRef) + ' Lon=' + str(lonRef) + ', Level = ' +str(level)
    # Use geocat.viz.util convenience function to set titles and labels
    gv.set_titles_and_labels(ax[iplot,j],
                         righttitle=Rtitle,                         
                         righttitlefontsize=15)  
    
    # Get the handles and labels from the legend
    handles, labels = ax[iplot,j].get_legend_handles_labels()
    
    # Remove duplicates from the handles and labels
    handles = handles[-3:]  # Keep only the first handle (MSE)
    labels = labels[-3:]  # Keep only the first label (MSE)
    
    # Update the legend with the filtered handles and labels
    ax[iplot,j].legend(handles, labels)
    ax[iplot,j].set_ylim(-0.75, 2)    
    ax[iplot,j].tick_params(axis='y', labelsize=15)
    
    plt.tight_layout()
    ax[iplot,j].hlines(y=0, xmin=-0.4, xmax=12.4, color='black', linewidth=0.8)

In [ ]:
path_composite = '/glade/derecho/scratch/aiyyer/mse/composites/refV'
plotsdir = '/glade/u/home/athornton/plots/projections/'

In [ ]:
itracks = [5, 3, 1]

# Selections:
levels = [850,600]
stormTracks = ['N', 'N', 'O'],['S', 'S', 'O']
Ltitle = ['Northern Track','Southern Track']

deltaLon = 15
deltaLat = 5

label = ['(a) ','(b) ','(c) '], ['(d) ','(e) ','(f) ']

In [ ]:
def lat_lon_bounds(stormTrack, lonRef, latRef, deltaLon):
    
    if (stormTrack == 'N'):
        latS = 17 #15.
        latN = 22.
    if (stormTrack == 'S' ):
        latS = 0. #5.
        latN = 17. #25.
    if (stormTrack == 'O' ):
        latS = 5.
        latN = 20. #25.

    lonW = lonRef - deltaLon
    lonE = lonRef + deltaLon
    return latS, latN, lonW, lonE

In [ ]:
def mse_projection(mse, mse_tend, EnerVar):
    variables = var_list(EnerVar)
    varNames = name_list(EnerVar)
    h_proj = []
    firstTime = True
    for variab, varNam in zip(variables, varNames):
        fileName = path_composite + str(level) + '/' + variab + '_Lag_Composite_RefV_' +  \
        str(level) + 'hPa' + '_track_point_' + str(itrack) + '.nc'    
        df = xr.open_dataset(fileName)
        lonRef = df.Reflon.values
        latRef = df.Reflat.values
        latS,latN,lonW,lonE = lat_lon_bounds(stormTrack,lonRef,latRef,deltaLon)
        forc = df[varNam].sel(latitude=slice(latN, latS),
                              longitude=slice(lonW, lonE),
                              Lag=0).metpy.quantify()

        numer = xr.dot(forc, mse_tend, dims=['latitude', 'longitude'], optimize=True)
        frac = (numer/denom)
        print(variab, "numer = ", numer.values , "  Projection = ", frac.values)
        h_proj.append(frac.values)

        if (firstTime):
            sum_forc = forc
            firstTime=False
        else:
            sum_forc = forc + sum_forc  # Add to previous value 

    residual = mse_tend - sum_forc
    numer = xr.dot(residual, mse_tend, dims=['latitude', 'longitude'], optimize=True)
    frac = (numer / denom)
    print('residual = ', numer.values, " Projection = ", frac.values)
    # add the residual to the proj array
    h_proj.append(frac.values)
    return h_proj

In [ ]:
# mse
def mse_func():
    variab = 'mse_vint'
    varNam = 'mse_vint_prime'  
    fileName = path_composite + str(level) + '/' + variab + '_Lag_Composite_RefV_' +  \
    str(level) + 'hPa' + '_track_point_' + str(itrack) + '.nc'
    #print(fileName)
    df = xr.open_dataset(fileName)
    lonRef = df.Reflon.values
    latRef = df.Reflat.values
    latS,latN,lonW,lonE = lat_lon_bounds(stormTrack,lonRef,latRef,deltaLon)
    mse = df[varNam].sel(latitude=slice(latN, latS), longitude=slice(lonW, lonE), Lag=0).metpy.quantify()    
 
    variab = 'hp_vint_tend'
    varNam = 'hp_vint_tend_prime'  
    fileName = path_composite + str(level) + '/' + variab + '_Lag_Composite_RefV_' +  \
    str(level) + 'hPa' + '_track_point_' + str(itrack) + '.nc'
    print(fileName)
    df = xr.open_dataset(fileName)
    lonRef = df.Reflon.values
    latRef = df.Reflat.values
    mse_tend = df[varNam].sel(latitude=slice(latN, latS), longitude=slice(lonW, lonE), Lag=0).metpy.quantify()
    print('ref lat/lon=', latRef, lonRef)

    mse.attrs['lonRef'] = lonRef
    mse.attrs['latRef'] = latRef
    return mse, mse_tend

In [ ]:
fig, ax = plt.subplots(3,2,constrained_layout=True)
# Set figure size (width, height) in inches
fig.set_size_inches((16,14))
plt.rcParams['savefig.dpi'] = 255

for j in range(0,2):
    for iplot in range(0,3):
        level = levels[j]
        itrack = itracks[iplot]
        stormTrack = stormTracks[j][iplot]
        mse, mse_tend = mse_func()
        denom = xr.dot(mse_tend,mse_tend, dims=['latitude','longitude'], optimize=True)
        h_proj = mse_projection(mse, mse_tend, 'h')
        s_proj = mse_projection(mse, mse_tend, 's')
        q_proj = mse_projection(mse, mse_tend, 'q')
    
    
        #print( iplot, denom.values)
        # zero out fluxes and residual since they do not belong here 
        s_proj[9:13] = [0., 0., 0., 0.]
        q_proj[9:13] = [0., 0., 0., 0.]
    
        lonRef = mse.attrs['lonRef'] 
        latRef = mse.attrs['latRef'] 
     
        #title = location[iplot]
        #title = location[iplot]
        #title = 'Northern Track'
        
        plotting()
        # Add the map subplot to the existing figure
        inset_ax = inset_axes(ax[iplot,j],
                        width="18%", # width = 30% of parent_bbox
                        height=1.0, # height : 1 inch
                        loc=2, #bbox_transform=ax.transAxes,
                        axes_class=cartopy.mpl.geoaxes.GeoAxes,
                        axes_kwargs=dict(map_projection=ccrs.PlateCarree()))
        minlon = lonRef - 30.
        maxlon = lonRef + 30.
        minlat = -5.
        maxlat = 30.
    
        #Set the extent or specific region for the inset map
        inset_ax.set_extent([minlon, maxlon, minlat, maxlat])  
    
        #Add coastlines and other features to the inset map
        inset_ax.add_feature(cfeature.COASTLINE, zorder=1)
        inset_ax.plot(lonRef, latRef, marker='o', markersize=5, color='red', transform=ccrs.PlateCarree())
    
gv.set_titles_and_labels(ax[0,0],
                         lefttitle=Ltitle[0],
                         lefttitlefontsize=15)
gv.set_titles_and_labels(ax[0,1],
                         lefttitle=Ltitle[1],
                         lefttitlefontsize=15)

#ax[0].set_ylabel('Contribution to Maintenance', fontsize=18)
ax[2,0].set_xlabel('Terms', fontsize=14)
ax[2,1].set_xlabel('Terms', fontsize=14)

ax[0,0].set_ylabel('Propagation Contribution', fontsize=14)
ax[1,0].set_ylabel('Propagation Contribution', fontsize=14)
ax[2,0].set_ylabel('Propagation Contribution', fontsize=14)

plt.subplots_adjust(top=0.925)  # Adjust 'top' to increase space
#plt.suptitle('Relative Contribution to Propagation', weight='bold', fontsize = 20)
#plt.savefig('propagation_projections.png', bbox_inches='tight')